# VulnLens local triage model (Kaggle)

Trains CodeBERT on PrimeVul, exports ONNX, and zips the artifact for download.

**Before running:** Settings → Accelerator → **GPU T4 x2**, Internet → **On**.
Kaggle requires phone verification for GPU and internet.

Start with `SMOKE = True` (about 10 minutes) to prove the pipeline works. Then set
`SMOKE = False` and *Save & Run All*; expect several hours.

**Honesty note:** PrimeVul is C/C++. These numbers are C/C++ function
classification, not Python performance. The Python measurement happens back on
your machine with `npm run eval:model`.


## 1. Get the code

Either push the repo to GitHub and set `REPO_URL`, or zip the repo (with a
top-level `vuln-lens/` folder) and upload it as a private Kaggle dataset, then
set `REPO_ZIP` to the path under `/kaggle/input/`.


In [ ]:
# ---- edit one of these ----
REPO_URL = ""   # e.g. "https://github.com/you/vuln-lens.git"
REPO_ZIP = ""   # e.g. "/kaggle/input/vuln-lens-code/vuln-lens.zip"

SMOKE = True    # flip to False for the real run
WORK = "/kaggle/working"

import os, pathlib, subprocess

assert REPO_URL or REPO_ZIP, "Set REPO_URL or REPO_ZIP above."
repo = pathlib.Path(WORK) / "vuln-lens"

if not repo.exists():
    if REPO_ZIP:
        subprocess.run(["unzip", "-q", REPO_ZIP, "-d", WORK], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo)], check=True)

if not (repo / "ml" / "kaggle" / "train_codebert.py").exists():
    matches = list(pathlib.Path(WORK).glob("*/ml/kaggle/train_codebert.py"))
    assert matches, "Could not find the repo after unzip/clone."
    repo = matches[0].parents[2]

os.chdir(repo)
print("repo:", repo)
print("smoke:", SMOKE)


## 2. Install training dependencies

Kaggle already ships PyTorch; this adds the rest.

In [ ]:
!pip install -q transformers datasets scikit-learn onnx onnxruntime


## 3. Prepare PrimeVul

Deduplicates across splits (test wins over validation over train) and writes
`train/valid/test.jsonl` plus a manifest.


In [ ]:
LIMIT = 2000 if SMOKE else 0
DATA = f"{WORK}/primevul-smoke" if SMOKE else f"{WORK}/primevul"
!python ml/data/prepare_primevul.py --out {DATA} --limit {LIMIT}


## 4. Fine-tune

Checkpoint selection uses validation PR-AUC. The script records recall at 1/5/10%
FPR, metrics at threshold 0.5, and at a threshold chosen for a 0.95 recall floor.


In [ ]:
OUT = f"{WORK}/codebert-smoke" if SMOKE else f"{WORK}/codebert-vuln"

if SMOKE:
    !python ml/kaggle/train_codebert.py --data {DATA} --output {OUT} --epochs 1 --max-train 500 --max-eval 200
else:
    !python ml/kaggle/train_codebert.py --data {DATA} --output {OUT} --epochs 3 --batch-size 16


## 5. Export ONNX

Verifies the exported graph against PyTorch logits and checks the int8 graph for
drift. Produces `model.onnx`, `vocab.json`, `merges.txt`, `labels.json`,
`model_card.json`, and `training_metrics.json`.


In [ ]:
ART = f"{WORK}/vuln-model"
!python ml/kaggle/export_onnx.py --checkpoint {OUT} --out {ART}


## 6. Inspect and download

In [ ]:
!ls -lh {ART}
!cd {WORK} && rm -f vuln-model.zip && zip -qr vuln-model.zip vuln-model && ls -lh vuln-model.zip
print("\nDownload /kaggle/working/vuln-model.zip from the Output panel on the right.")


## 7. Back on your machine

```bash
cd ~/vuln-lens
unzip ~/Downloads/vuln-model.zip -d ~/models
npm install onnxruntime-node
npm run eval:model -- --model ~/models/vuln-model
```

Then in VS Code set `vulnLens.localModel.enabled` to `true`,
`vulnLens.localModel.path` to the artifact directory, and run
**VulnLens: Check Local Model**.

Read `eval/MODEL_RESULTS.md`. Issue coverage is the number that matters: an issue
not inside a high-scoring function never reaches the verifier.
